In [ ]:
import requests
import pandas as pd
import time
##API사용하여 ADHD 관련 네이버 지식iN 질문 1000개 크롤링
# 1. 네이버 API 정보 설정
# 본인의 Client ID와 Client Secret으로 교체해야 합니다.
CLIENT_ID = "YOUR_CLIENT_ID" 
CLIENT_SECRET = "YOUR_CLIENT_SECRET"

# API 요청을 보낼 기본 URL
BASE_URL = "https://openapi.naver.com/v1/search/kin.json"

# 2. 크롤링 설정
QUERY = "Your Search Query"  # 검색어
TARGET_COUNT = 1000  # 목표 크롤링 개수
DISPLAY_COUNT = 100  # 한 번에 가져올 개수 (최대 100)

# 3. API 호출 및 데이터 수집
print(f"'{QUERY}'에 대한 네이버 지식iN 크롤링을 시작합니다. 목표: {TARGET_COUNT}개")

all_results = []
start_index = 1

while start_index <= TARGET_COUNT:
    try:
        # API 요청에 필요한 파라미터 설정
        params = {
            "query": QUERY,
            "display": DISPLAY_COUNT,
            "start": start_index,
            "startDate": "2025-01-01",
            "endDate": "2025-12-31"
        }

        # API 요청 헤더 설정
        headers = {
            "X-Naver-Client-Id": CLIENT_ID,
            "X-Naver-Client-Secret": CLIENT_SECRET
        }

        # API 요청 보내기
        response = requests.get(BASE_URL, params=params, headers=headers)
        
        # 응답 상태 확인
        response.raise_for_status() # 오류 발생 시 예외를 발생시킴

        # JSON 데이터 파싱
        data = response.json()
        items = data.get('items', [])

        if not items:
            print("더 이상 검색 결과가 없습니다. 크롤링을 중단합니다.")
            break

        all_results.extend(items)
        
        print(f"현재까지 수집된 질문 수: {len(all_results)} / {data.get('total', 0)}개")

        # 다음 페이지 요청을 위해 시작 인덱스 업데이트
        start_index += DISPLAY_COUNT
        
        # API 호출 제한을 피하기 위해 약간의 딜레이를 줍니다.
        time.sleep(0.5)

    except requests.exceptions.HTTPError as e:
        print(f"HTTP 에러가 발생했습니다: {e}")
        print("Client ID와 Secret이 올바른지, API 할당량을 초과하지 않았는지 확인하세요.")
        break
    except Exception as e:
        print(f"알 수 없는 오류가 발생했습니다: {e}")
        break

print(f"\n총 {len(all_results)}개의 질문을 성공적으로 수집했습니다.")






In [ ]:
# 4. 수집된 데이터를 DataFrame으로 변환 및 저장
if all_results:
    # 필요한 정보(질문 제목, 링크, 요약 내용, 날짜)만 추출
    df = pd.DataFrame(all_results)
    df = df[['title', 'link', 'description']]

    # HTML 태그 제거 (예: <b>ADHD</b> -> ADHD)
    df['title'] = df['title'].str.replace(r'<[^>]+>', '', regex=True)
    df['description'] = df['description'].str.replace(r'<[^>]+>', '', regex=True)

    # 파일로 저장
    filename = f"naver_kin_{QUERY}_{TARGET_COUNT}.csv"
    df.to_csv(filename, index=False, encoding='utf-8-sig')

    print(f"\n크롤링 결과가 '{filename}' 파일로 저장되었습니다.")
    print("저장된 데이터 샘플:")
    print(df.head())
else:
    print("\n수집된 데이터가 없어 파일을 저장하지 않았습니다.")

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from tqdm import tqdm
# 1. 원본 CSV 파일 경로를 지정합니다.
file_path = r"Your file path here"  

# 2. 날짜가 추가될 새로운 파일의 이름을 지정합니다. Add Date
output_file_path = r"Your output file path here"

try:
    df = pd.read_csv(file_path)
    print(f"'{file_path}' 파일을 성공적으로 읽었습니다. 총 {len(df)}개의 데이터가 있습니다.")
except FileNotFoundError:
    print(f"오류: '{file_path}' 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")
    exit()

dates = []
print("\n각 링크를 방문하여 날짜 수집을 시작합니다...")

for row in tqdm(df.itertuples(), total=df.shape[0], desc="날짜 수집 중"):
    link_url = row.link
    
    try:
        response = requests.get(link_url, headers={'User-Agent': 'Mozilla/5.0'})
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # --- [점검 및 수정 완료된 핵심 부분] ---
        # 'infoItem' 클래스를 가지면서, 내부에 'blind' 클래스를 가진 자식 요소를 포함하는 span 태그를 선택
        date_element = soup.select_one("span.infoItem:has(span.blind)")
        
        date_text = "N/A" # 기본값
        if date_element:
            # '작성일' 텍스트를 제거하고 순수한 날짜만 추출
            date_text = date_element.get_text(strip=True).replace("작성일", "").strip()
            dates.append(date_text)
        else:
            dates.append("N/A (날짜 요소 없음)")
        # ------------------------------------

        time.sleep(0.1)

    except requests.exceptions.RequestException as e:
        print(f"\nURL 접속 오류: {link_url} - {e}")
        dates.append("오류 발생")
    except Exception as e:
        print(f"\n알 수 없는 오류 발생: {link_url} - {e}")
        dates.append("오류 발생")

if len(dates) == len(df):
    df['date'] = dates
    print("\n날짜 수집이 완료되어 'date' 컬럼을 추가했습니다.")
    
    # 컬럼 순서 재정렬
    if 'description' in df.columns:
        df = df[['date', 'title', 'link', 'description']]
    else:
        df = df[['date', 'title', 'link']]

    df.to_csv(output_file_path, index=False, encoding='utf-8-sig')
    
    print(f"\n작업 완료! 데이터가 '{output_file_path}' 파일에 저장되었습니다.")
    print("\n--- 최종 데이터 샘플 (상위 5개) ---")
    print(df.head())
else:
    print("\n오류: 수집된 날짜의 개수가 원본 데이터의 행 개수와 일치하지 않습니다.")



In [ ]:
import pandas as pd
from konlpy.tag import Okt
from collections import Counter
import re

# 1. 분석할 파일 경로를 지정합니다.
# 날짜가 추가된 최신 파일을 사용합니다.
file_path = r"Your file path here"  # 예: "C:/Users/ghldn/Projects/ADHD2/naver_kin_ADHD_1000_1201_with_dates.csv"

try:
    df = pd.read_csv(file_path)
    print(f"'{file_path}' 파일을 성공적으로 읽었습니다.")
except FileNotFoundError:
    print(f"오류: '{file_path}' 파일을 찾을 수 없습니다. 경로를 다시 확인해주세요.")
    exit()

# 2. 'description' 컬럼의 모든 텍스트를 하나의 문자열로 합치기
# 결측치(NaN)가 있을 경우를 대비하여 처리
df.dropna(subset=['description'], inplace=True)
text = " ".join(df['description'].tolist())

# 3. 텍스트 정제: 한글, 영문, 숫자만 남기고 특수문자 제거
text = re.sub(r'[^ㄱ-ㅎㅏ-ㅣ가-힣a-zA-Z0-9\s]', '', text)

# 4. 형태소 분석기를 이용한 토큰화 (명사 추출)
okt = Okt()
nouns = okt.nouns(text)

# 5. 불용어 제거
# 직접 불용어 사전을 정의합니다. 분석 목적에 따라 추가/삭제할 수 있습니다.
stop_words = ['그', '저', '것', '수', '등', '더', '이', '있', '하', '되', '을', '를', '은', '는', '에', '의', 
    '가', '와', '과', '도', '으로', '해서', '합니다', '한다', '같은', '어떤', '또한', '하지만', '/','아','인','.','..','들',
    '어','식','없','는데','게','알','질','볼','건','은데','습니다','게','같','어떻게','는데','때','부터',
    '보','아','못','아','기','좋','안','님','잘','을까요','대','적','네요','입니다','해야','해','다고','았','없','않','인데','할'
    '못하','했','어','거나','주','세요','라고','니','어서','아서','여','좀','겠','보이','는지','아요','다는','정도','해요',
    '될','떨어질','으면','만','서','아니','맞','성','됩니다','데','혹시','는데요','에게','지만','할까요','까지','셨','듣',
    '한다고','면서','라','번','그런','함','고요','께서','비해','이게','봤','니까','걸','아서요','라면','아고','한데','건지',
    '한테','줄','어떻','된','이제','라는','이랑','봐','을지','그냥','그래서','거든요','왔','그런데','둔','다니','아무래도',
    '분','건가요','그렇','하나','인데요','후','그러','보입니다','질까','던','또','죠','크','진','던지','스럽','시키','던데','라서','이후',
    '해서요','며','이렇게','된다고','한가요','라도','엔','오','애','참','나왔','드','더니','거니','냐고','두','본','서요','~','갔',
    '야','곤','씩','이거','보였','셔서','한다는','았었','줘야','려','저희','도록','그때','으니','.?','하고','(',')','됐','지요','졌','치',
    '.','?',',','고','지','한','로','전','혹시','면','제','었','로','말','다','거','일','할','에서','시''다가','싶','=', '%','-','&','*','@','#','!','^','+','=','|','`','~']

# 한 글자 단어 및 불용어 리스트에 포함된 단어 제거
keywords = [word for word in nouns if len(word) > 1 and word not in stop_words]

# 6. 단어 빈도 분석
counter = Counter(keywords)

# 7. 가장 빈도가 높은 상위 N개 키워드 추출
N = 20
top_keywords = counter.most_common(N)

print(f"\n--- 'description' 컬럼의 상위 {N}개 키워드 ---")
for i, (word, count) in enumerate(top_keywords):
    print(f"{i+1:2d}위: {word} (등장 횟수: {count}회)")



In [ ]:
import requests
import pandas as pd
import time
from bs4 import BeautifulSoup
#위 빈도에 따라 다빈도 순으로 추출하기
# 네이버 OpenAPI 인증정보
headers_api = {
    "X-Naver-Client-Id": "YOUR_CLIENT_ID",
    "X-Naver-Client-Secret": "YOUR_CLIENT_SECRET"
}

headers_web = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36"
}


search_queries = ["ADHD 키워드1", "ADHD 키워드2", "ADHD 키워드3", "ADHD 키워드4", "ADHD 키워드5", "ADHD 키워드6", "ADHD 키워드7", "ADHD 키워드8", "ADHD 키워드9", "ADHD 키워드10"]
all_results = []

# 질문 전문 후보 셀렉터(사이트 개편 상황 대비 다중 시도)
QUESTION_SELECTORS = [
    "div.questionDetail._endContents._endContentsText",
    "div#question-content",
    "div.questionDetail",
    "div.c-heading__content",
    "div._questionContents",
    "div.question-content",
]

# 답변 전문 셀렉터(당신이 준 셀렉터 + 몇 가지 백업)
ANSWER_SELECTORS = [
    "div.answerDetail._endContents._endContentsText",   # ← 당신이 준 정답 셀렉터
    "div.c-heading-answer__content",
    "div._endContents._endContentsText",
    "div.answer-content",
]

def html_to_text(elem):
    """HTML 요소에서 텍스트만 깔끔히 추출"""
    if not elem:
        return ""
    # 링크/해시태그/불필요한 공백 정리
    text = elem.get_text(" ", strip=True)
    # 과도한 공백 줄이기
    return " ".join(text.split())

def extract_question_answer(page_html):
    """질문/답변 전문 텍스트 추출 (셀렉터 후보를 순차 시도)"""
    soup = BeautifulSoup(page_html, "html.parser")

    # 질문
    question_text = ""
    for sel in QUESTION_SELECTORS:
        node = soup.select_one(sel)
        question_text = html_to_text(node)
        if len(question_text) >= 30:   # 최소 길이 기준(짧은 안내/광고 등 걸러내기)
            break

    # 답변 (최상위 답변 하나 추출; 여러 개 원하면 select로 loop)
    answer_text = ""
    for sel in ANSWER_SELECTORS:
        node = soup.select_one(sel)
        answer_text = html_to_text(node)
        if len(answer_text) >= 30:
            break

    return question_text, answer_text

all_rows = []

for query in search_queries:
    print(f"🔍 검색어: {query}")
    # 최대 1000개까지(네이버 정책): start = 1, 101, 201, ... , 901
    for start in range(1, 1000, 100):
        api_url = f"https://openapi.naver.com/v1/search/kin.json?query={query}&display=100&start={start}"
        r = requests.get(api_url, headers=headers_api, timeout=15)
        if r.status_code != 200:
            print(f"❌ API 실패: {r.status_code}")
            break

        items = r.json().get("items", [])
        if not items:
            break

        for it in items:
            title_text = BeautifulSoup(it["title"], "html.parser").get_text(" ", strip=True)
            question_snippet = BeautifulSoup(it["description"], "html.parser").get_text(" ", strip=True)
            link = it["link"]

            # 링크 접속해서 질문/답변 전문 추출
            question_full, answer_full = "", ""
            try:
                resp = requests.get(link, headers=headers_web, timeout=15)
                if resp.status_code == 200:
                    question_full, answer_full = extract_question_answer(resp.text)
            except Exception as e:
                print(f"  ⚠ 페이지 크롤링 오류: {e}")

            all_rows.append({
                "query": query,
                "title": title_text,
                "question_snippet": question_snippet,  # API 요약(키워드 포함 문장)
                "question_full": question_full,        # 🔹 질문 전문
                "answer_full": answer_full,            # 🔹 답변 전문(최상위)
                "link": link
            })

        print(f"✅ {start} ~ {start+99} 수집")
        time.sleep(1.2)  # 너무 빠르면 차단 가능 → 적당히 쉬기

# CSV 저장
df = pd.DataFrame(all_rows)
df.to_csv("Your output file path here", index=False, encoding="utf-8-sig")
print("📁 CSV 저장 완료: Your output file name here")